<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip install -q datasets huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

df = dataset["train"].to_pandas()

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

In [ ]:
print(df.shape)

(2414248, 21)


In [ ]:
df.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'query_hash_id',
 'query_char_count',
 'query_token_count',
 'window_start',
 'window_end',
 'impressions_90d',
 'clicks_90d',
 'impressions_last30',
 'clicks_last30',
 'impressions_prev30',
 'clicks_prev30',
 'avg_position_90d',
 'avg_position_last30',
 'avg_position_prev30',
 'content_total_impressions_90d',
 'content_visible_query_count',
 'rare_query_count',
 'rare_impressions_share',
 'anonymized_impressions_share']

In [ ]:
df.iloc[0]

,0
client_hash_id,client_08a6a72ff48e62c0
content_hash_id,content_447894f2faf0d2bc
query_hash_id,query_58b1b001f839d699
query_char_count,17
query_token_count,3
window_start,2026-04-02
window_end,2026-06-30
impressions_90d,11
clicks_90d,0
impressions_last30,0


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

One row represents the search performance of a single **content–query pair** for one client over a **90-day observation window**. The window is defined by the `window_start` and `window_end` columns. For example, one row covers the period from **2026-04-02** to **2026-06-30**. This unit of analysis allows us to study how each content page performs for a specific search query over a fixed time period.

In [ ]:
print("Dataset Shape:", df.shape)

print("\nObservation Window:")
print(df[["window_start", "window_end"]].head())

print("\nUnique Window Start Dates:", df["window_start"].nunique())
print("Unique Window End Dates:", df["window_end"].nunique())

Dataset Shape: (2414248, 21)

Observation Window:
  window_start  window_end
0   2026-04-02  2026-06-30
1   2026-04-02  2026-06-30
2   2026-04-02  2026-06-30
3   2026-04-02  2026-06-30
4   2026-04-02  2026-06-30

Unique Window Start Dates: 1
Unique Window End Dates: 1


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

### Features
I will use the following features to describe search performance:

- impressions_90d
- clicks_90d
- avg_position_90d
- content_total_impressions_90d
- content_visible_query_count

These features are available before making a content review decision and can help rank refresh opportunities.

### Label / Proxy
The dataset does not contain a direct refresh label. Instead, I will use a **proxy target** by ranking content-query pairs based on their search performance (for example, lower clicks, lower visibility, or poorer average position).

### Context
The following fields provide context but are not model features:

- client_hash_id
- content_hash_id
- query_hash_id
- window_start
- window_end

These identify the content, query, client, and observation window.

### Excluded
I will exclude:

- rare_impressions_share
- anonymized_impressions_share

These are excluded because they are anonymized summary measures and are not essential for my initial feature set.

In [ ]:
# Display the selected feature, context, and excluded fields

selected_columns = [
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "window_start",
    "window_end",
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "content_total_impressions_90d",
    "content_visible_query_count",
    "rare_impressions_share",
    "anonymized_impressions_share"
]

df[selected_columns].head()

,client_hash_id,content_hash_id,query_hash_id,window_start,window_end,impressions_90d,clicks_90d,avg_position_90d,content_total_impressions_90d,content_visible_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,2026-04-02,2026-06-30,11,0,10.818182,1466,14,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,2026-04-02,2026-06-30,13,0,1.769231,1466,14,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,2026-04-02,2026-06-30,16,0,23.562500,1466,14,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,2026-04-02,2026-06-30,55,0,2.200000,1466,14,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,2026-04-02,2026-06-30,14,0,3.428571,1466,14,0.043656,0.725102


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verify it with queries

The following queries verify the key parts of my data contract:

- The grain of the dataset (one row = one content-query pair).
- The number of rows and the observation window.
- Missing values in the main feature columns.

In [ ]:
import pandas as pd

# 1. Verify the grain
print("Duplicate content-query pairs:")
duplicates = df.duplicated(subset=["content_hash_id", "query_hash_id", "window_start", "window_end"]).sum()
print(duplicates)

# 2. Row count and date span
print("\nRow count:")
print(len(df))

print("\nDate span:")
print(df[["window_start", "window_end"]].agg(["min", "max"]))

# 3. Missing values in key features
print("\nMissing values:")
print(df[["impressions_90d", "clicks_90d", "avg_position_90d"]].isna().sum())

Duplicate content-query pairs:
0

Row count:
2414248

Date span:
    window_start  window_end
min   2026-04-02  2026-06-30
max   2026-04-02  2026-06-30

Missing values:
impressions_90d     0
clicks_90d          0
avg_position_90d    0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset provides observed search performance for content-query pairs over 90-day windows. It supports analysis and ranking of content opportunities, but it cannot explain why performance changed or prove that refreshing content will improve results. It also does not include external factors such as Google algorithm updates, competitor actions, seasonality, or content quality. Therefore, any conclusions should be treated as decision support rather than causal evidence.

In [ ]:
print("Earliest window start:", df["window_start"].min())
print("Latest window end:", df["window_end"].max())

print("\nDataset size:", len(df))

Earliest window start: 2026-04-02
Latest window end: 2026-06-30

Dataset size: 2414248
